In [3]:
import torch
import torch.nn as nn
import math

# -----------------------
# 1. Toy data
# -----------------------
sentences = [
    "the cat sat on the mat",
    "the dog lay on the rug",
    "a cat chased a mouse",
    "the mouse ran away",
    "dogs and cats are animals",
    "the rug is on the floor",
    "the mat is near the door",
    "animals live in the forest",
    "the cat and dog play",
    "the mouse eats cheese"
]


# -----------------------
# 2. Tokenization & vocab
# -----------------------
def build_vocab(sentences):
    tokens = set()
    for s in sentences:
        tokens.update(s.split())
    vocab = ["<pad>", "<unk>"] + sorted(list(tokens))
    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for w, i in stoi.items()}
    return vocab, stoi, itos


def encode(sentences, stoi, max_len=None):
    if max_len is None:
        max_len = max(len(s.split()) for s in sentences)
    encoded = []
    for s in sentences:
        ids = [stoi.get(w, stoi["<unk>"]) for w in s.split()]
        ids = ids + [stoi["<pad>"]] * (max_len - len(ids))
        encoded.append(ids)
    return torch.tensor(encoded, dtype=torch.long), max_len


# -----------------------
# 3. Positional encoding
# -----------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x


# -----------------------
# 4. Scaled dot-product attention
# -----------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn, V)
    return output, attn


# -----------------------
# 5. Multi-head attention
# -----------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # (batch, heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        # concat heads
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.W_o(out)
        return out, attn


# -----------------------
# 6. Transformer encoder block
# -----------------------
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model=64, num_heads=2, d_ff=128, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + Add & Norm
        attn_out, attn_weights = self.mha(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward + Add & Norm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x, attn_weights


# -----------------------
# 7. Putting it together
# -----------------------
if __name__ == "__main__":
    vocab, stoi, itos = build_vocab(sentences)
    encoded, max_len = encode(sentences, stoi)
    print("Input token IDs:\n", encoded)

    d_model = 64
    num_heads = 2

    embed = nn.Embedding(len(vocab), d_model)
    pos_enc = PositionalEncoding(d_model)
    encoder_block = TransformerEncoderBlock(d_model=d_model, num_heads=num_heads)

    x = embed(encoded)           # (batch, seq_len, d_model)
    x = pos_enc(x)               # add positional encoding
    contextual, attn_weights = encoder_block(x)

    print("\nFinal contextual embeddings (batch, seq_len, d_model):")
    print(contextual.shape)

    # Show attention heatmap for first sentence, first head
    attn_first = attn_weights[0, 0]  # (seq_len, seq_len)
    print("\nAttention weights (sentence 0, head 0):")
    print(attn_first)

    print("\nTokens for sentence 0:")
    print(sentences[0].split())


Input token IDs:
 tensor([[29,  7, 28, 24, 29, 21],
        [29, 11, 19, 24, 29, 27],
        [ 2,  7,  9,  2, 22,  0],
        [29, 22, 26,  6,  0,  0],
        [12,  3,  8,  5,  4,  0],
        [29, 27, 18, 24, 29, 15],
        [29, 21, 18, 23, 29, 13],
        [ 4, 20, 17, 29, 16,  0],
        [29,  7,  3, 11, 25,  0],
        [29, 22, 14, 10,  0,  0]])

Final contextual embeddings (batch, seq_len, d_model):
torch.Size([10, 6, 64])

Attention weights (sentence 0, head 0):
tensor([[0.0962, 0.2001, 0.1987, 0.2231, 0.0865, 0.1954],
        [0.2192, 0.2073, 0.1507, 0.1245, 0.1984, 0.0998],
        [0.0653, 0.1042, 0.2094, 0.2727, 0.0629, 0.2855],
        [0.1747, 0.2354, 0.1725, 0.1508, 0.1547, 0.1119],
        [0.1074, 0.2119, 0.1616, 0.2464, 0.1002, 0.1725],
        [0.1415, 0.0716, 0.2614, 0.1699, 0.1178, 0.2379]],
       grad_fn=<SelectBackward0>)

Tokens for sentence 0:
['the', 'cat', 'sat', 'on', 'the', 'mat']
